# Raw Data to IC

This notebook teaches the Phase 1 AI-DOP pipeline:

```text
Aardvark sample_data_final.pkl
-> inspect observation sources
-> pretrained Aardvark encoder
-> IC-like physical weather state at t = 0
-> Zarr output
-> table / CSV view
```

Important: in this project, `sample_data_final.pkl` is our **raw observation sample** for Phase 1, but technically it is already Aardvark-preprocessed model-ready data. It is not untouched BUFR, NetCDF, or satellite files.

## 1. Imports and Display Setup

This cell imports the reusable Phase 1 modules. The code path uses CUDA if available, otherwise CPU. The notebook does not use MPS for Phase 1 because our target environment is CUDA-first.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from weather_research.aardvark_initial_conditions import (
    choose_compute_device,
    generate_aardvark_initial_condition_from_file,
)
from weather_research.aardvark_observations import (
    DEFAULT_AARDVARK_SAMPLE_PATH,
    load_aardvark_observation_sample,
)
from weather_research.weather_state_schema import AARDVARK_STATE_VARIABLES
from weather_research.weather_state_tables import initial_condition_dataset_to_table
from weather_research.weather_state_xarray import (
    initial_condition_to_dataset,
    save_initial_condition_zarr,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

DEVICE = choose_compute_device("auto")
OUTPUT_DIR = Path("outputs")
IC_ZARR_PATH = OUTPUT_DIR / "aardvark_initial_condition.zarr"
IC_TABLE_CSV_PATH = OUTPUT_DIR / "aardvark_initial_condition_table.csv"

print(f"Device selected: {DEVICE}")
print(f"Sample path: {DEFAULT_AARDVARK_SAMPLE_PATH}")

Device selected: cpu
Sample path: /Users/ewencheung/Documents/GitHub/Weather-Research/Reference-Repo-aardvark-weather-public-main/data/sample_data_final.pkl


## 2. Load the Raw Observation Sample

This cell reads `sample_data_final.pkl`. The loaded object is a dictionary with sections:

- `assimilation`: observation inputs used by the encoder.
- `forecast`: gridded context used by the forecast processor later.
- `downscaling`: station target/context information used by the decoder later.
- `y_target`: station target values supplied in the sample.

For Phase 1, the important input is `data["assimilation"]`. That is what gets mapped into the IC-like state.

In [ ]:
sample = load_aardvark_observation_sample(DEFAULT_AARDVARK_SAMPLE_PATH)
data = sample.payload
assimilation = data["assimilation"]

print("Loaded:", sample.path)
print("Top-level sections:", list(data.keys()))
print("Assimilation keys:", len(assimilation))

Loaded: /Users/ewencheung/Documents/GitHub/Weather-Research/Reference-Repo-aardvark-weather-public-main/data/sample_data_final.pkl
Top-level sections: ['assimilation', 'forecast', 'downscaling', 'y_target']
Assimilation keys: 26


## 3. Raw Data Source Table

This table answers: **which source gives what information?**

The table uses one row per source because satellites, stations, marine platforms, and radiosonde profiles do not share the same shape. The source IDs such as `satellite_1` are teaching labels for this notebook.

In [ ]:
hadisd_variables = ["tas", "tds", "psl", "u", "v"]

source_rows = [
    {
        "source_id": "satellite_1",
        "source_name": "HIRS",
        "family": "satellite radiance",
        "value_key": "hirs_current",
        "location_key": "hirs_x_current",
        "shape": tuple(assimilation["hirs_current"].shape),
        "channels_or_variables": 26,
        "information_given": "infrared sounding radiance channels",
    },
    {
        "source_id": "satellite_2",
        "source_name": "AMSU-A",
        "family": "satellite radiance",
        "value_key": "amsua_current",
        "location_key": "amsua_x_current",
        "shape": tuple(assimilation["amsua_current"].shape),
        "channels_or_variables": 13,
        "information_given": "microwave temperature sounding channels",
    },
    {
        "source_id": "satellite_3",
        "source_name": "AMSU-B / MHS",
        "family": "satellite radiance",
        "value_key": "amsub_current",
        "location_key": "amsub_x_current",
        "shape": tuple(assimilation["amsub_current"].shape),
        "channels_or_variables": 12,
        "information_given": "microwave humidity sounding channels",
    },
    {
        "source_id": "satellite_4",
        "source_name": "IASI",
        "family": "satellite radiance",
        "value_key": "iasi_current",
        "location_key": "iasi_x_current",
        "shape": tuple(assimilation["iasi_current"].shape),
        "channels_or_variables": 52,
        "information_given": "infrared hyperspectral sounding channels",
    },
    {
        "source_id": "satellite_5",
        "source_name": "ASCAT",
        "family": "satellite scatterometer",
        "value_key": "ascat_current",
        "location_key": "ascat_x_current",
        "shape": tuple(assimilation["ascat_current"].shape),
        "channels_or_variables": 17,
        "information_given": "surface wind-related ocean scatterometer features",
    },
    {
        "source_id": "satellite_6",
        "source_name": "GRIDSAT-like sample",
        "family": "satellite gridded imagery",
        "value_key": "sat_current",
        "location_key": "sat_x_current",
        "shape": tuple(assimilation["sat_current"].shape),
        "channels_or_variables": 2,
        "information_given": "gridded satellite image channels",
    },
    {
        "source_id": "station_1",
        "source_name": "HadISD",
        "family": "land station",
        "value_key": "y_context_hadisd_current",
        "location_key": "x_context_hadisd_current",
        "shape": [tuple(x.shape) for x in assimilation["y_context_hadisd_current"]],
        "channels_or_variables": hadisd_variables,
        "information_given": "surface station variables: temperature, dew point, pressure, wind",
    },
    {
        "source_id": "station_2",
        "source_name": "ICOADS",
        "family": "marine station / ship / buoy",
        "value_key": "icoads_current",
        "location_key": "icoads_x_current",
        "shape": tuple(assimilation["icoads_current"].shape),
        "channels_or_variables": 5,
        "information_given": "marine surface observations",
    },
    {
        "source_id": "profile_1",
        "source_name": "IGRA",
        "family": "radiosonde profile",
        "value_key": "igra_current",
        "location_key": "igra_x_current",
        "shape": tuple(assimilation["igra_current"].shape),
        "channels_or_variables": 24,
        "information_given": "upper-air profile observations",
    },
]

raw_source_table = pd.DataFrame(source_rows)
raw_source_table

,source_id,source_name,family,value_key,location_key,shape,channels_or_variables,information_given
0,satellite_1,HIRS,satellite radiance,hirs_current,hirs_x_current,"(1, 360, 181, 26)",26,infrared sounding radiance channels
1,satellite_2,AMSU-A,satellite radiance,amsua_current,amsua_x_current,"(1, 180, 360, 13)",13,microwave temperature sounding channels
2,satellite_3,AMSU-B / MHS,satellite radiance,amsub_current,amsub_x_current,"(1, 360, 181, 12)",12,microwave humidity sounding channels
3,satellite_4,IASI,satellite radiance,iasi_current,iasi_x_current,"(1, 360, 181, 52)",52,infrared hyperspectral sounding channels
4,satellite_5,ASCAT,satellite scatterometer,ascat_current,ascat_x_current,"(1, 360, 181, 17)",17,surface wind-related ocean scatterometer features
5,satellite_6,GRIDSAT-like sample,satellite gridded imagery,sat_current,sat_x_current,"(1, 2, 514, 200)",2,gridded satellite image channels
6,station_1,HadISD,land station,y_context_hadisd_current,x_context_hadisd_current,"[(1, 8719), (1, 8617), (1, 8016), (1, 8721), (...","[tas, tds, psl, u, v]","surface station variables: temperature, dew po..."
7,station_2,ICOADS,marine station / ship / buoy,icoads_current,icoads_x_current,"(1, 5, 12000)",5,marine surface observations
8,profile_1,IGRA,radiosonde profile,igra_current,igra_x_current,"(1, 24, 1375)",24,upper-air profile observations


## 4. Quick Numeric Summary of Raw Sources

This cell gives a small sanity check for the source tensors. It does not explain the meteorology yet; it helps us confirm that the inputs are finite tensors with expected shapes before encoding.

In [ ]:
summary_rows = []
for row in source_rows:
    value = assimilation[row["value_key"]]
    if isinstance(value, list):
        first_tensor = value[0]
        tensor_count = len(value)
    else:
        first_tensor = value
        tensor_count = 1

    array = first_tensor.detach().cpu().float()
    summary_rows.append(
        {
            "source_id": row["source_id"],
            "source_name": row["source_name"],
            "tensor_count": tensor_count,
            "first_tensor_shape": tuple(first_tensor.shape),
            "mean": float(array.mean()),
            "std": float(array.std()),
            "min": float(array.min()),
            "max": float(array.max()),
        }
    )

raw_numeric_summary = pd.DataFrame(summary_rows)
raw_numeric_summary

,source_id,source_name,tensor_count,first_tensor_shape,mean,std,min,max
0,satellite_1,HIRS,1,"(1, 360, 181, 26)",NaN,NaN,NaN,NaN
1,satellite_2,AMSU-A,1,"(1, 180, 360, 13)",NaN,NaN,NaN,NaN
2,satellite_3,AMSU-B / MHS,1,"(1, 360, 181, 12)",NaN,NaN,NaN,NaN
3,satellite_4,IASI,1,"(1, 360, 181, 52)",NaN,NaN,NaN,NaN
4,satellite_5,ASCAT,1,"(1, 360, 181, 17)",NaN,NaN,NaN,NaN
5,satellite_6,GRIDSAT-like sample,1,"(1, 2, 514, 200)",NaN,NaN,NaN,NaN
6,station_1,HadISD,5,"(1, 8719)",NaN,NaN,NaN,NaN
7,station_2,ICOADS,1,"(1, 5, 12000)",NaN,NaN,NaN,NaN
8,profile_1,IGRA,1,"(1, 24, 1375)",NaN,NaN,NaN,NaN


## 5. Generate the IC-Like State

This is the core Phase 1 step.

The encoder fuses all the observation sources into a gridded physical state. After this step, the output is no longer separated by satellite or station. It becomes:

```text
time, latitude, longitude, weather variables
```

The 24 variables are Aardvark's IC-like state variables.

In [ ]:
ic = generate_aardvark_initial_condition_from_file(
    DEFAULT_AARDVARK_SAMPLE_PATH,
    device_preference="auto",
)

print("IC shape:", ic.values.shape)
print("Variable count:", len(ic.variable_names))
print("Variables:", ic.variable_names)
print("Finite values:", np.isfinite(ic.values).all())

IC shape: (121, 240, 24)
Variable count: 24
Variables: ('u10', 'v10', 't2m', 'mslp', 'z200', 'z500', 'z700', 'z850', 'q200', 'q500', 'q700', 'q850', 't200', 't500', 't700', 't850', 'u200', 'u500', 'u700', 'u850', 'v200', 'v500', 'v700', 'v850')
Finite values: True


## 6. Convert IC to xarray Dataset

xarray gives names to the dimensions and variables. This makes the IC easier to inspect, save, and later evaluate in a WeatherBench-style format.

In [ ]:
ic_dataset = initial_condition_to_dataset(
    ic,
    time=np.datetime64("2019-01-01T00:00:00"),
)

ic_dataset

<xarray.Dataset> Size: 3MB
Dimensions:                  (time: 1, latitude: 121, longitude: 240)
Coordinates:
  * time                     (time) datetime64[ns] 8B 2019-01-01
  * latitude                 (latitude) float32 484B -90.0 -88.5 ... 88.5 90.0
  * longitude                (longitude) float32 960B 0.0 1.502 ... 357.5 359.0
Data variables: (12/24)
    10m_u_component_of_wind  (time, latitude, longitude) float32 116kB -6.04 ...
    10m_v_component_of_wind  (time, latitude, longitude) float32 116kB -5.218...
    2m_temperature           (time, latitude, longitude) float32 116kB 221.0 ...
    mean_sea_level_pressure  (time, latitude, longitude) float32 116kB 1.018e...
    geopotential_200         (time, latitude, longitude) float32 116kB 1.046e...
    geopotential_500         (time, latitude, longitude) float32 116kB 4.906e...
    ...                       ...
    u_component_of_wind_700  (time, latitude, longitude) float32 116kB -6.12 ...
    u_component_of_wind_850  (time, latitude, longitude) float32 116kB -6.408...
    v_component_of_wind_200  (time, latitude, longitude) float32 116kB 2.678 ...
    v_component_of_wind_500  (time, latitude, longitude) float32 116kB -4.53 ...
    v_component_of_wind_700  (time, latitude, longitude) float32 116kB -4.91 ...
    v_component_of_wind_850  (time, latitude, longitude) float32 116kB -4.802...
Attributes:
    title:    Aardvark IC-like physical weather state
    source:   /Users/ewencheung/Documents/GitHub/Weather-Research/Reference-R...
    note:     Generated from Aardvark model-ready observations. This is not A...

## 7. Show the IC at t = 0 as a Table

This is the table view you asked for. Each row is one grid point at `t = 0`.

The columns are:

```text
time, latitude, longitude, u10, v10, t2m, mslp, ...
```

This is the mapped IC-like state after the satellite and station observations have been fused by the encoder.

In [ ]:
ic_table_preview = initial_condition_dataset_to_table(
    ic_dataset,
    max_rows=20,
)

ic_table_preview

,time,latitude,longitude,u10,v10,t2m,mslp,z200,z500,z700,z850,q200,q500,q700,q850,t200,t500,t700,t850,u200,u500,u700,u850,v200,v500,v700,v850
0,2019-01-01,-90.0,0.000000,-6.040300,-5.218417,221.012344,101811.882812,104639.656250,49055.890625,26227.472656,12443.799805,0.000002,0.000069,0.000099,0.000443,185.755722,233.138580,228.941406,237.018372,-1.713733,-4.479140,-6.120273,-6.407934,2.678020,-4.530162,-4.909699,-4.801717
1,2019-01-01,-90.0,1.502092,-6.194963,-5.332687,220.266571,101804.421875,104692.765625,49070.390625,26233.974609,12423.879883,0.000002,0.000091,0.000153,0.000421,185.985977,232.990356,229.291428,237.370239,-1.929119,-4.011099,-6.218165,-6.198985,2.049605,-4.693857,-5.080583,-5.009847
2,2019-01-01,-90.0,3.004184,-5.810433,-5.601009,220.171967,101765.750000,104752.101562,49112.871094,26246.611328,12391.808594,0.000003,0.000121,0.000313,0.000334,185.845566,233.028854,229.647034,237.892334,-1.985192,-3.622713,-6.284207,-5.726382,1.542182,-5.199195,-5.618549,-4.874613
3,2019-01-01,-90.0,4.506276,-5.422510,-5.539505,221.525391,101732.312500,104654.382812,49074.054688,26222.433594,12401.450195,0.000002,0.000074,0.000112,0.000440,185.952560,233.057724,229.898712,237.849030,-1.817472,-4.299076,-5.668107,-5.783020,1.069730,-5.343931,-4.968904,-5.142608
4,2019-01-01,-90.0,6.008368,-5.627412,-5.672140,220.898285,101734.984375,104705.601562,49094.414062,26233.234375,12391.876953,0.000002,0.000089,0.000149,0.000432,186.073181,232.964905,230.218933,238.078156,-2.064169,-3.839499,-5.739130,-5.593662,0.355897,-5.571411,-5.039501,-5.363938
5,2019-01-01,-90.0,7.510460,-5.286582,-5.913101,220.714539,101711.710938,104755.109375,49136.632812,26249.648438,12370.678711,0.000003,0.000113,0.000319,0.000376,185.922638,232.968323,230.540695,238.481339,-2.210409,-3.452417,-5.807534,-5.136049,-0.144270,-6.018719,-5.442071,-5.317115
6,2019-01-01,-90.0,9.012552,-4.994364,-6.346338,221.823929,101740.203125,104701.343750,49067.769531,26231.562500,12405.776367,0.000002,0.000072,0.000146,0.000455,186.073441,232.963058,229.937347,238.059021,-2.566715,-3.844489,-5.247514,-5.435688,0.667949,-6.519804,-5.701514,-5.974446
7,2019-01-01,-90.0,10.514645,-5.151556,-6.362607,221.380981,101738.812500,104747.750000,49093.925781,26242.769531,12396.929688,0.000002,0.000085,0.000157,0.000445,186.205414,232.913651,230.316330,238.281555,-2.838724,-3.414857,-5.199061,-5.136583,-0.113547,-6.763853,-5.610839,-6.071361
8,2019-01-01,-90.0,12.016736,-4.760072,-6.514188,221.230408,101714.734375,104785.718750,49138.062500,26260.982422,12377.921875,0.000003,0.000106,0.000305,0.000415,186.145325,232.897003,230.700790,238.713989,-3.051840,-3.026389,-5.180985,-4.628007,-0.600251,-7.146358,-5.746363,-6.071369
9,2019-01-01,-90.0,13.518828,-4.688778,-6.948963,221.623474,101790.828125,104709.843750,49065.582031,26235.423828,12427.509766,0.000002,0.000074,0.000174,0.000479,185.923172,233.082306,229.673431,237.756699,-3.110428,-3.258376,-4.958015,-5.154159,1.020379,-7.425690,-6.363648,-6.554814


## 8. Save Zarr and CSV Outputs

This cell writes two useful artifacts:

- Zarr: keeps the gridded IC dataset with named dimensions.
- CSV: gives a spreadsheet-style table view for inspection.

Generated files are written under `outputs/`, which is ignored by git.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

save_initial_condition_zarr(
    ic,
    IC_ZARR_PATH,
    time=np.datetime64("2019-01-01T00:00:00"),
    overwrite=True,
)

full_ic_table = initial_condition_dataset_to_table(ic_dataset, max_rows=None)
full_ic_table.to_csv(IC_TABLE_CSV_PATH, index=False)

print("Saved Zarr:", IC_ZARR_PATH)
print("Saved CSV:", IC_TABLE_CSV_PATH)
print("Full IC table rows:", len(full_ic_table))

Saved Zarr: outputs/aardvark_initial_condition.zarr
Saved CSV: outputs/aardvark_initial_condition_table.csv
Full IC table rows: 29040


## 9. Reopen the Saved Zarr

This verifies that the saved Zarr output can be loaded again. This matters because later phases will consume the saved IC instead of recomputing it every time.

In [ ]:
reopened = xr.open_zarr(IC_ZARR_PATH)
print(reopened)
print("2m_temperature mean:", float(reopened["2m_temperature"].mean()))
print("2m_temperature min:", float(reopened["2m_temperature"].min()))
print("2m_temperature max:", float(reopened["2m_temperature"].max()))

<xarray.Dataset> Size: 3MB
Dimensions:                  (time: 1, latitude: 121, longitude: 240)
Coordinates:
  * time                     (time) datetime64[ns] 8B 2019-01-01
  * latitude                 (latitude) float32 484B -90.0 -88.5 ... 88.5 90.0
  * longitude                (longitude) float32 960B 0.0 1.502 ... 357.5 359.0
Data variables: (12/24)
    10m_u_component_of_wind  (time, latitude, longitude) float32 116kB dask.array<chunksize=(1, 121, 240), meta=np.ndarray>
    10m_v_component_of_wind  (time, latitude, longitude) float32 116kB dask.array<chunksize=(1, 121, 240), meta=np.ndarray>
    2m_temperature           (time, latitude, longitude) float32 116kB dask.array<chunksize=(1, 121, 240), meta=np.ndarray>
    geopotential_200         (time, latitude, longitude) float32 116kB dask.array<chunksize=(1, 121, 240), meta=np.ndarray>
    geopotential_500         (time, latitude, longitude) float32 116kB dask.array<chunksize=(1, 121, 240), meta=np.ndarray>
    geopotential_700  

## 10. Equivalent Terminal Commands

The notebook cells above use Python functions directly. These are the equivalent commands from the terminal.

In [ ]:
print("Generate IC Zarr:")
print("uv run --no-sync weather-research raw-to-ic --device auto --output outputs/aardvark_initial_condition.zarr --time 2019-01-01T00:00:00")
print()
print("Show IC table:")
print("uv run --no-sync weather-research show-ic-table --input outputs/aardvark_initial_condition.zarr --rows 20")
print()
print("Save IC table CSV:")
print("uv run --no-sync weather-research show-ic-table --input outputs/aardvark_initial_condition.zarr --rows 20 --output-csv outputs/aardvark_initial_condition_table.csv")

Generate IC Zarr:
uv run --no-sync weather-research raw-to-ic --device auto --output outputs/aardvark_initial_condition.zarr --time 2019-01-01T00:00:00

Show IC table:
uv run --no-sync weather-research show-ic-table --input outputs/aardvark_initial_condition.zarr --rows 20

Save IC table CSV:
uv run --no-sync weather-research show-ic-table --input outputs/aardvark_initial_condition.zarr --rows 20 --output-csv outputs/aardvark_initial_condition_table.csv
